# Cake or Dog — interactive notebook

This notebook is intended for interactive work with the Cake-or-Dog classifier.

Main goals:
- model training and evaluation
- error visualization
- interactive predictions

## 0. Setup

Make sure that Jupyter is running in the same venv as the project.

In [ ]:
# !uv sync

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from ipywidgets import interact  # type: ignore
from PIL import Image

from cakeordog.data import (  # type: ignore
    CATEGORIES,
    load_single_image,
    load_split_parallel,
)
from cakeordog.model import (  # type: ignore
    load_model,
    predict_label,
    save_model,
    train_svm_gridsearch,
)

print(sys.executable)

## 1. Paths to data and model

In [ ]:
DATA_TRAIN = "../data/train"
DATA_TEST = "../data/test"
MODEL_OUT_DUMMY = "../models/notebook_model.joblib"
MODEL_OUT = "../models/svc_muffin_chihuahua.joblib"
ANTI_ALIASING = False

## 2. Loading test data

In [ ]:
x_test, y_test = load_split_parallel(DATA_TEST, anti_aliasing=ANTI_ALIASING)

print(f"Amount of processed test files: {len(x_test)}")

### 2.1 Show processed image

In [ ]:
x = x_test[0]
print(x)
x = x.reshape(32, 32, 3)
x = x.astype(np.uint8)
im = Image.fromarray(x)
im = im.convert("RGB")
im = im.resize((256, 256))

display(im)

## 3. Model training

SVM + GridSearch is used.
The result contains the best model and metrics on the test set.

***Warning***: For training you must download dataset from https://www.kaggle.com/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification/data

In [ ]:
# Amount of images for training.
# Use function load_split_parallel without that variable for taking all data
MAX_AMOUNT = 128

x_train, y_train = load_split_parallel(
    DATA_TRAIN, MAX_AMOUNT, anti_aliasing=ANTI_ALIASING
)
print(f"Amount of processed train files: {len(x_train)}")

In [ ]:
result = train_svm_gridsearch(
    data_train=x_train,
    labels_train=y_train,
    data_test=x_test,
    labels_test=y_test,
)

print(result.best_params, result.test_accuracy)

## 4. Saving the model

In [ ]:
Path(MODEL_OUT_DUMMY).parent.mkdir(exist_ok=True)
save_model(result.best_model, MODEL_OUT_DUMMY)
print(MODEL_OUT_DUMMY)

## 5. Batch predict

Run the model on the test dataset and collect the results into a DataFrame

### Our model

In [ ]:
model = load_model(MODEL_OUT_DUMMY)

rows = []
for img in Path(DATA_TEST).rglob("*.jpg"):
    x = load_single_image(str(img), anti_aliasing=ANTI_ALIASING)
    pred = predict_label(model, x)
    rows.append(
        {
            "path": str(img),
            "true": img.parent.name,
            "pred": CATEGORIES[pred],
        }
    )

df = pd.DataFrame(rows)
df.head()

#### Accuracy and confusion matrix

In [ ]:
(df.true == df.pred).mean()

In [ ]:
pd.crosstab(df.true, df.pred, normalize="index")

### Pretrained model

In [ ]:
model = load_model(MODEL_OUT)

rows = []
for img in Path(DATA_TEST).rglob("*.jpg"):
    x = load_single_image(str(img), anti_aliasing=True)
    pred = predict_label(model, x)
    rows.append(
        {
            "path": str(img),
            "true": img.parent.name,
            "pred": CATEGORIES[pred],
        }
    )

df = pd.DataFrame(rows)
df.head()

#### Accuracy and confusion matrix

In [ ]:
(df.true == df.pred).mean()

In [ ]:
pd.crosstab(df.true, df.pred, normalize="index")

## 6. Error inspection

Interactive slider for viewing incorrectly classified images

In [ ]:
errors = df[df.true != df.pred].reset_index(drop=True)
len(errors)

In [ ]:
MAX_HEIGHT = 300  # pixels


@interact(i=(0, len(errors) - 1))
def show_error(i: int) -> None:
    row = errors.iloc[i]
    path_str = str(row.path) if isinstance(row.path, Path) else row.path

    img: Image.Image = Image.open(path_str)

    # Save proportions, limit height
    w, h = img.size
    new_w = int(w * MAX_HEIGHT / h)
    resized_img: Image.Image = img.resize((new_w, MAX_HEIGHT))

    display(resized_img)
    print("true:", row.true)
    print("pred:", row.pred)

## 7. Single image prediction

In [ ]:
MAX_HEIGHT = 300

for path in [
    DATA_TEST + "/chihuahua/img_0_580.jpg",
    DATA_TEST + "/muffin/img_0_602.jpg",
]:
    file_path = Path(path)

    img_single: Image.Image = Image.open(file_path)

    w, h = img_single.size
    if h > MAX_HEIGHT:
        new_w = int(w * MAX_HEIGHT / h)
        img_single = img_single.resize((new_w, MAX_HEIGHT))

    display(img_single)
    x = load_single_image(path)
    pred = predict_label(model, x)
    display(CATEGORIES[pred])

### 8. Our friends

Let us know what our friends are

Before running script put photos of your friends to data/friends folder

In [ ]:
MAX_HEIGHT = 450

model = load_model(MODEL_OUT)

folder_path = Path("../data/friends")
image_files = list(folder_path.iterdir())

result_count = {CATEGORIES[0]: 0, CATEGORIES[1]: 0}
results = []

for file_path in image_files:
    img_friend: Image.Image = Image.open(file_path)

    w, h = img_friend.size
    if h > MAX_HEIGHT:
        new_w = int(w * MAX_HEIGHT / h)
        img_friend = img_friend.resize((new_w, MAX_HEIGHT))

    x = load_single_image(str(file_path), anti_aliasing=False)
    pred = predict_label(model, x)
    results.append((img_friend, CATEGORIES[pred]))
    result_count[CATEGORIES[pred]] += 1

print(result_count)

for img_friend, pred in results:
    display(img_friend)
    display(pred)